# 05 — Agentic RAG: Smart Research Assistant

Combine **LangGraph** with **RAG** to build a self-correcting research assistant.

### What makes it "agentic"?
A regular RAG pipeline always retrieves, regardless of the question.
This agent **decides**:
1. Does this question need retrieval at all?
2. Is the retrieved context good enough to answer?
3. If not — rewrite the query and try again (up to 2 retries)

### Graph flow
```
START → route_query
  ├─ (needs retrieval)  → retrieve → assess_context
  │                           ├─ (context OK)     → generate_answer → END
  │                           └─ (context poor, retry < 2) → rewrite_query → retrieve → ...
  │                           └─ (context poor, retry >= 2) → generate_answer → END
  └─ (no retrieval needed) → generate_answer → END
```

> **Requires:** `GROQ_API_KEY` in a `.env` file

## 1. Install Dependencies

In [ ]:
# !pip install langgraph langchain langchain-community langchain-groq langchain-huggingface faiss-cpu python-dotenv

## 2. Imports & Setup

In [ ]:
import os
from typing import TypedDict, List, Literal
from pathlib import Path
from dotenv import load_dotenv

from langchain_core.documents import Document
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage
from langgraph.graph import StateGraph, START, END

load_dotenv()
print("GROQ_API_KEY set:", bool(os.getenv("GROQ_API_KEY")))

## 3. Build the Knowledge Base

In [ ]:
DATA_DIR = Path("../data")

# Load all 4 topic files
raw_docs = []
for filepath in DATA_DIR.glob("*.txt"):
    text = filepath.read_text(encoding="utf-8")
    raw_docs.append(Document(
        page_content=text,
        metadata={"source": filepath.name, "topic": filepath.stem}
    ))
print(f"Loaded {len(raw_docs)} documents")

# Split into chunks
splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=60)
chunks   = splitter.split_documents(raw_docs)
print(f"Created {len(chunks)} chunks")

# Build FAISS vector store
print("Building vector store...")
embedder    = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = FAISS.from_documents(chunks, embedder)
retriever   = vectorstore.as_retriever(search_kwargs={"k": 4})
print("Vector store ready!")

## 4. Initialise the LLM

In [ ]:
llm = ChatGroq(model="llama3-8b-8192", temperature=0.1)
print("Groq LLM ready")

## 5. Define the Agent State

The state is the shared object that every node reads and updates.
It carries the question, retrieved documents, retry count, and final answer.

In [ ]:
class ResearchState(TypedDict):
    question:         str         # original user question
    active_query:     str         # current query (may be rewritten)
    documents:        List[Document]  # retrieved chunks
    context:          str         # formatted context string
    context_score:    float       # quality score of retrieved context (0-1)
    answer:           str         # final answer
    needs_retrieval:  bool        # agent decision: retrieve or answer directly?
    retry_count:      int         # number of query rewrites so far

## 6. Define the Agent Nodes

Each node is a Python function that reads the state and returns a partial update.

In [ ]:
RETRIEVAL_KEYWORDS = {
    "what", "who", "when", "where", "how", "why", "which",
    "explain", "describe", "tell me", "define", "list"
}

def route_query(state: ResearchState) -> dict:
    """Node 1 — decide if retrieval is needed based on question keywords."""    words = set(state["question"].lower().split())
    needs_retrieval = bool(words & RETRIEVAL_KEYWORDS)
    active_query    = state["question"]
    print(f"  🧭 route_query → needs_retrieval={needs_retrieval}")
    return {"needs_retrieval": needs_retrieval, "active_query": active_query}

In [ ]:
def retrieve(state: ResearchState) -> dict:
    """Node 2 — search the vector store using the active query."""    query = state["active_query"]
    print(f"  🔍 retrieve → '{query}'")
    docs    = retriever.invoke(query)
    context = "\n\n".join(
        f"[Source: {d.metadata.get('source', '?')}]\n{d.page_content}"
        for d in docs
    )
    return {"documents": docs, "context": context}

In [ ]:
def assess_context(state: ResearchState) -> dict:
    """Node 3 — ask the LLM to rate how well the context answers the question."""    question = state["question"]
    context  = state["context"]

    prompt = (
        f"Rate how well this context answers the question, on a scale of 0.0 to 1.0.\n\n"
        f"Question: {question}\n\n"
        f"Context (first 600 chars):\n{context[:600]}\n\n"
        f"Reply with ONLY a decimal number like 0.7. Nothing else."
    )
    response = llm.invoke([HumanMessage(content=prompt)])
    try:
        score = float("".join(c for c in response.content if c.isdigit() or c == "."))
        score = max(0.0, min(1.0, score))
    except Exception:
        score = 0.5
    print(f"  📊 assess_context → score={score:.2f}")
    return {"context_score": score}

In [ ]:
def rewrite_query(state: ResearchState) -> dict:
    """Node 4 — rephrase the question to retrieve better context."""    original = state["question"]
    attempt  = state["retry_count"] + 1

    prompt = (
        f"The original question '{original}' did not retrieve good context.\n"
        f"Rewrite it as a clearer, more specific search query. This is attempt {attempt}.\n"
        f"Reply with ONLY the rewritten query, nothing else."
    )
    response = llm.invoke([HumanMessage(content=prompt)])
    new_query = response.content.strip().strip('"')
    print(f"  ✏️  rewrite_query → '{new_query}'")
    return {"active_query": new_query, "retry_count": attempt}

In [ ]:
def generate_answer(state: ResearchState) -> dict:
    """Node 5 — generate the final answer from context (or directly)."""    question = state["question"]
    context  = state.get("context", "")

    if context:
        prompt = (
            f"Answer the following question using the context provided.\n"
            f"Be clear and concise. Mention the source if relevant.\n\n"
            f"Context:\n{context}\n\n"
            f"Question: {question}\n\nAnswer:"
        )
    else:
        prompt = f"Answer this question directly: {question}"

    response = llm.invoke([HumanMessage(content=prompt)])
    answer   = response.content.strip()
    retries  = state.get("retry_count", 0)
    score    = state.get("context_score", 0.0)
    print(f"  ✅ generate_answer → done (retries={retries}, ctx_score={score:.2f})")
    return {"answer": answer}

## 7. Conditional Routing Functions

These functions look at the current state and return the **name of the next node**.

In [ ]:
CONTEXT_THRESHOLD = 0.35   # minimum acceptable context quality score
MAX_RETRIES       = 2      # max query rewrites before giving up

def decide_after_routing(state: ResearchState) -> Literal["retrieve", "generate_answer"]:
    """After route_query: go to retriever or answer directly."""    return "retrieve" if state["needs_retrieval"] else "generate_answer"

def decide_after_assessment(
    state: ResearchState
) -> Literal["generate_answer", "rewrite_query"]:
    """After assess_context: accept the context or rewrite the query."""    score   = state.get("context_score", 0.0)
    retries = state.get("retry_count", 0)
    if score >= CONTEXT_THRESHOLD or retries >= MAX_RETRIES:
        return "generate_answer"
    return "rewrite_query"

print("Routing functions defined")

## 8. Build and Compile the Graph

In [ ]:
builder = StateGraph(ResearchState)

# Register nodes
builder.add_node("route_query",     route_query)
builder.add_node("retrieve",        retrieve)
builder.add_node("assess_context",  assess_context)
builder.add_node("rewrite_query",   rewrite_query)
builder.add_node("generate_answer", generate_answer)

# Entry point
builder.add_edge(START, "route_query")

# After routing: retrieve OR answer directly
builder.add_conditional_edges(
    "route_query",
    decide_after_routing,
    {"retrieve": "retrieve", "generate_answer": "generate_answer"}
)

# After retrieval: always assess context quality
builder.add_edge("retrieve", "assess_context")

# After assessment: accept OR rewrite query (loop back to retrieve)
builder.add_conditional_edges(
    "assess_context",
    decide_after_assessment,
    {"generate_answer": "generate_answer", "rewrite_query": "rewrite_query"}
)

# After rewriting: retrieve again
builder.add_edge("rewrite_query", "retrieve")

# Terminal edge
builder.add_edge("generate_answer", END)

graph = builder.compile()
print("Graph compiled!")

## 9. Visualise the Graph *(optional)*

In [ ]:
from IPython.display import Image, display
try:
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception as e:
    print(f"Visualisation skipped: {e}")

## 10. Run the Research Assistant

In [ ]:
def ask(question: str) -> str:
    """Run the full agentic RAG pipeline for a question."""    print(f"\n{'═'*58}")
    print(f"  Question: {question}")
    print(f"{'─'*58}")

    result = graph.invoke({
        "question":        question,
        "active_query":    question,
        "documents":       [],
        "context":         "",
        "context_score":   0.0,
        "answer":          "",
        "needs_retrieval": True,
        "retry_count":     0,
    })

    print(f"{'─'*58}")
    print(f"  Answer: {result['answer']}")
    print(f"{'═'*58}")
    return result["answer"]

In [ ]:
# Questions requiring retrieval (uses vector store)
_ = ask("What is a black hole and how does it form?")
_ = ask("How much protein do I need if I exercise regularly?")
_ = ask("What Python technique avoids re-computing expensive functions?")

In [ ]:
# A general question that skips retrieval
_ = ask("What is 2 + 2?")

## 11. Key Takeaways

| Concept | What you learned |
|---------|-----------------|
| Agentic routing | Agent decides whether retrieval is needed at all |
| Context assessment | LLM scores its own retrieved context for quality |
| Self-correcting loop | Poor context → rewrite query → retrieve again |
| MAX_RETRIES guard | Prevents infinite loops when context is always poor |
| LangGraph state | All agent decisions flow through a single typed state object |

### RAG Series Complete!

| # | Notebook | Key concept |
|---|----------|-------------|
| 01 | RAG Foundations | Cosine similarity from scratch |
| 02 | Vector Store RAG | FAISS — save/load/search |
| 03 | Q&A Pipeline | LangChain LCEL chain |
| 04 | Multi-Doc RAG | ChromaDB + metadata filtering |
| 05 | Agentic RAG | LangGraph self-correcting retrieval |